# CSV Basics — 04: Handling Dirty and Malformed Data

## What you will learn
Real-world CSV files are messy. This notebook teaches you to handle:
1. Malformed records — Spark's three error modes
2. `columnNameOfCorruptRecord` — capturing bad rows
3. Mixed types in columns — coercion strategies
4. Extra and missing columns
5. Duplicate headers and whitespace issues
6. Building a robust CSV ingestion pipeline with quarantine


In [ ]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

In [ ]:
import pathlib

# CSV with various data quality issues
dirty_csv = """order_id,customer,amount,quantity,order_date
1,Alice,100.50,2,2024-01-15
2,Bob,INVALID_AMOUNT,1,2024-01-16
3,Carol,300.00,abc,2024-01-17
4,Dave,400.00,4,NOT_A_DATE
5,Eve,500.00,2,2024-01-19
extra_col,Grace,200,3,2024-01-20,UNEXPECTED_EXTRA_COLUMN
7,,150.00,1,2024-01-21
8,Henry,350.00,2,2024-01-22
"""

pathlib.Path(f"{DATA_DIR}/dirty.csv").write_text(dirty_csv)
print("Created dirty.csv with various data quality issues")
print("Issues:")
print("  Row 2: amount = 'INVALID_AMOUNT' (should be double)")
print("  Row 3: quantity = 'abc' (should be int)")
print("  Row 4: order_date = 'NOT_A_DATE' (should be date)")
print("  Row 6: extra column at end")
print("  Row 7: empty customer name")

## Step 1 — Three Error Modes

Spark has three modes for handling corrupt records:

| Mode | Behavior | Use when |
|---|---|---|
| `PERMISSIVE` | Set bad values to null, save raw row in `_corrupt_record` | Production — don't lose data |
| `DROPMALFORMED` | Silently drop rows with any parse error | Quick analysis — don't care about bad rows |
| `FAILFAST` | Throw exception on first bad record | Data validation — must be perfect |


In [ ]:
schema = StructType([
    StructField("order_id",    IntegerType()),
    StructField("customer",    StringType()),
    StructField("amount",      DoubleType()),
    StructField("quantity",    IntegerType()),
    StructField("order_date",  DateType()),
])

# PERMISSIVE mode (default) — bad values become null
df_permissive = spark.read.csv(
    f"{DATA_DIR}/dirty.csv",
    header=True, schema=schema, mode="PERMISSIVE"
)
print("PERMISSIVE mode:")
df_permissive.show(truncate=False)
print(f"  Rows: {df_permissive.count()} (all rows kept, bad values = null)")

In [ ]:
# DROPMALFORMED — silently drop any row with a parse error
df_drop = spark.read.csv(
    f"{DATA_DIR}/dirty.csv",
    header=True, schema=schema, mode="DROPMALFORMED"
)
print("DROPMALFORMED mode:")
df_drop.show(truncate=False)
print(f"  Rows: {df_drop.count()} (bad rows silently dropped)")
print("  ⚠️  Warning: you won't know how many rows were dropped!")

In [ ]:
# FAILFAST — throw exception on first bad record
print("FAILFAST mode:")
try:
    df_fail = spark.read.csv(
        f"{DATA_DIR}/dirty.csv",
        header=True, schema=schema, mode="FAILFAST"
    )
    df_fail.count()   # trigger read
    print("  No errors found")
except Exception as e:
    print(f"  Exception raised: {type(e).__name__}")
    print("  Use this mode in data validation pipelines where data MUST be clean")

## Step 2 — Capturing Corrupt Records with columnNameOfCorruptRecord

The best production approach: capture ALL bad rows in a special column
so you can quarantine and investigate them later.


In [ ]:
# Schema with extra column for corrupt records
schema_with_corrupt = StructType([
    StructField("order_id",         IntegerType()),
    StructField("customer",         StringType()),
    StructField("amount",           DoubleType()),
    StructField("quantity",         IntegerType()),
    StructField("order_date",       DateType()),
    StructField("_corrupt_record",  StringType()),   # ← captures raw bad rows
])

df_quarantine = spark.read.csv(
    f"{DATA_DIR}/dirty.csv",
    header=True,
    schema=schema_with_corrupt,
    mode="PERMISSIVE",
    columnNameOfCorruptRecord="_corrupt_record"
)

print("All rows with corrupt record captured:")
df_quarantine.show(truncate=False)

# Split into good and bad
df_good = df_quarantine.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
df_bad  = df_quarantine.filter(F.col("_corrupt_record").isNotNull())

print(f"\nGood rows : {df_good.count()}")
print(f"Bad rows  : {df_bad.count()}")
print()
print("Bad rows (quarantined):")
df_bad.select("_corrupt_record").show(truncate=False)

## Step 3 — Handling Extra and Missing Columns


In [ ]:
# CSV with inconsistent column counts
inconsistent_csv = """id,name,value
1,Alice,100
2,Bob,200,EXTRA_COL_HERE
3,Carol
4,Dave,400,ANOTHER_EXTRA,AND_MORE"""

pathlib.Path(f"{DATA_DIR}/inconsistent.csv").write_text(inconsistent_csv)

schema_inc = StructType([
    StructField("id",    IntegerType()),
    StructField("name",  StringType()),
    StructField("value", IntegerType()),
    StructField("_corrupt_record", StringType()),
])

df_inc = spark.read.csv(
    f"{DATA_DIR}/inconsistent.csv",
    header=True, schema=schema_inc,
    mode="PERMISSIVE",
    columnNameOfCorruptRecord="_corrupt_record"
)
print("Inconsistent column counts:")
df_inc.show(truncate=False)
print()
print("Extra columns → row captured as corrupt")
print("Missing columns → null values for missing fields")

## Step 4 — Whitespace and Header Issues


In [ ]:
# CSV with whitespace problems
whitespace_csv = """ id , name , value 
 1 , Alice , 100 
 2 , Bob , 200 
 3 , Carol , 300 """

pathlib.Path(f"{DATA_DIR}/whitespace.csv").write_text(whitespace_csv)

# Without trim — column names have spaces, values have spaces
df_no_trim = spark.read.csv(f"{DATA_DIR}/whitespace.csv",
                             header=True, inferSchema=True)
print("Without trimming:")
print("Column names:", df_no_trim.columns)
df_no_trim.show()

# With trim — clean names and values
df_trimmed = spark.read.csv(
    f"{DATA_DIR}/whitespace.csv",
    header=True, inferSchema=True,
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True
)
print("\nWith trimming:")
print("Column names:", df_trimmed.columns)
df_trimmed.show()

# Fix column names too
df_clean = df_trimmed.toDF(*[c.strip() for c in df_trimmed.columns])
print("\nWith trimmed column names:")
print("Column names:", df_clean.columns)
df_clean.show()

## Step 5 — Robust Ingestion Pipeline with Quarantine

This is the production-ready pattern for CSV ingestion:
1. Read with `columnNameOfCorruptRecord`
2. Split into good and bad
3. Write good to main table
4. Write bad to quarantine table for investigation


In [ ]:
import pathlib

# Production ingestion function
def ingest_csv(input_path, schema, output_good, output_bad):
    """
    Robust CSV ingestion with quarantine for bad records.
    Returns (good_count, bad_count)
    """
    # Add corrupt record column to schema
    full_schema = StructType(schema.fields + [
        StructField("_corrupt_record", StringType())
    ])

    # Read with PERMISSIVE + corrupt column
    df_raw = spark.read.csv(
        input_path,
        header=True,
        schema=full_schema,
        mode="PERMISSIVE",
        columnNameOfCorruptRecord="_corrupt_record",
        ignoreLeadingWhiteSpace=True,
        ignoreTrailingWhiteSpace=True,
    )

    # Split
    df_good = df_raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
    df_bad  = df_raw.filter(F.col("_corrupt_record").isNotNull()) \
                    .select("_corrupt_record",
                            F.current_timestamp().alias("_ingested_at"),
                            F.lit(str(input_path)).alias("_source_file"))

    # Write
    df_good.write.mode("overwrite").parquet(output_good)
    df_bad.write.mode("overwrite").parquet(output_bad)

    return df_good.count(), df_bad.count()

schema = StructType([
    StructField("order_id",   IntegerType()),
    StructField("customer",   StringType()),
    StructField("amount",     DoubleType()),
    StructField("quantity",   IntegerType()),
    StructField("order_date", DateType()),
])

good_count, bad_count = ingest_csv(
    f"{DATA_DIR}/dirty.csv",
    schema,
    f"{DATA_DIR}/ingested/good",
    f"{DATA_DIR}/ingested/bad",
)

print(f"Ingestion complete:")
print(f"  Good records : {good_count}")
print(f"  Bad records  : {bad_count}")
print()
print("Bad records (quarantined for investigation):")
spark.read.parquet(f"{DATA_DIR}/ingested/bad").show(truncate=False)

## Summary: Handling Dirty CSV Data

### Error mode selection
```python
# Production: capture bad rows, don't lose data
mode="PERMISSIVE", columnNameOfCorruptRecord="_corrupt_record"

# Quick analysis: just skip bad rows
mode="DROPMALFORMED"

# Data validation: must be perfect
mode="FAILFAST"
```

### Quarantine pattern (production standard)
```python
schema_with_corrupt = StructType(business_schema.fields + [
    StructField("_corrupt_record", StringType())
])
df = spark.read.csv(path, schema=schema_with_corrupt,
                    mode="PERMISSIVE",
                    columnNameOfCorruptRecord="_corrupt_record")

good = df.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")
bad  = df.filter(col("_corrupt_record").isNotNull())
```
